# Sentence Reconstruction

The purpose of this project is to take in input a sequence of words corresponding to a random permutation of a given english sentence, and reconstruct the original sentence.

The otuput can be either produced in a single shot, or through an iterative (autoregressive) loop generating a single token at a time.


CONSTRAINTS:
* No pretrained model can be used.
* The neural network models should have less the 20M parameters.
* No postprocessing should be done (e.g. no beamsearch)
* You cannot use additional training data.


BONUS PARAMETERS:

A bonus of 0-2 points will be attributed to incentivate the adoption of models with a low number of parameters.

# Dataset

The dataset is composed by sentences taken from the generics_kb dataset of hugging face. We restricted the vocabolary to the 10K most frequent words, and only took sentences making use of this vocabulary.

In [ ]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.


Download the dataset

In [ ]:
from datasets import load_dataset
from keras.layers import TextVectorization
import tensorflow as tf
import numpy as np

np.random.seed(42)
ds = load_dataset('generics_kb',trust_remote_code=True)['train']

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Filter row with length greater than 8.


In [ ]:
ds = ds.filter(lambda row: len(row["generic_sentence"].split(" ")) > 8 )
corpus = [ '<start> ' + row['generic_sentence'].replace(","," <comma>") + ' <end>' for row in ds ]
corpus = np.array(corpus)

Filter:   0%|          | 0/1020868 [00:00<?, ? examples/s]

Create a tokenizer and Detokenizer

In [ ]:
tokenizer=TextVectorization( max_tokens=10000, standardize="lower_and_strip_punctuation", encoding="utf-8",) #con il max prende le piu frequenti. ordina i token del vocab dal piu frequente al meno frequente
tokenizer.adapt(corpus)

class TextDetokenizer:
    def __init__(self, vectorize_layer):
        self.vectorize_layer = vectorize_layer
        vocab = self.vectorize_layer.get_vocabulary()
        self.index_to_word = {index: word for index, word in enumerate(vocab)}

    def __detokenize_tokens(self, tokens):
        def check_token(t):
          if t == 3:
            s="<start>"
          elif t == 2:
            s="<end>"
          elif t == 7:
            s="<comma>"
          else:
            s=self.index_to_word.get(t, '[UNK]')
          return s

        return ' '.join([ check_token(token) for token in tokens if token != 0])

    def __call__(self, batch_tokens):
       return [self.__detokenize_tokens(tokens) for tokens in batch_tokens]


detokenizer = TextDetokenizer( tokenizer )
sentences = tokenizer( corpus ).numpy()


Remove from corpus the sentences where any unknow word appears

In [ ]:
mask = np.sum( (sentences==1) , axis=1) >= 1
original_data = np.delete( sentences, mask , axis=0)

In [ ]:
original_data.shape

(241236, 28)

Shuffle the sentences

In [ ]:
# shifted_data = np.roll(original_data, 1, axis=1)

In [ ]:
from tensorflow.keras.utils import Sequence

class DataGenerator(Sequence):
    def __init__(self, data, training=True, batch_size=32, shuffle=True):

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.training = training
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        data_batch = np.array([self.data[k] for k in indexes])
        # shifted_result = np.array([shifted_data[k] for k in indexes])

        # copy of ordered sequences
        result = np.copy(data_batch)
        shifted_result = np.roll(result, -1, axis=1)

        #shuffle only the relevant positions for each batch
        for i in range(data_batch.shape[0]):
          np.random.shuffle(data_batch[i,1:data_batch[i].argmin() - 1])

        if(self.training):
          return [data_batch, result], shifted_result
        else:
          return [data_batch, data_batch], result

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
# train_generator = DataGenerator(original_data[:220000])
# test_generator = DataGenerator(original_data[220000:])
# x, y = test_generator.__getitem__(1)
# x = detokenizer(x[0])
# y = detokenizer(y)

# for i in range(7):
#   print("original: ", y[i])
#   print("shuffled: ", x[i])
#   print("\n")

original:  wind speed is a reflection of the air pressure gradients <end> <start>
shuffled:  <start> the speed pressure reflection of a air wind gradients is <end>


original:  visible light makes up a fraction of all electromagnetic energy <end> <start>
shuffled:  <start> up visible light of makes fraction electromagnetic energy a all <end>


original:  most women experience a menstrual period four to six weeks after a miscarriage <end> <start>
shuffled:  <start> six to menstrual miscarriage women a most after weeks experience a four period <end>


original:  all weasels have scent glands <comma> but some are more powerful than others <end> <start>
shuffled:  <start> more powerful but some glands have weasels <comma> scent all are others than <end>


original:  some women can develop the technique of achieving ejaculation <end> <start>
shuffled:  <start> achieving can of the ejaculation women develop technique some <end>


original:  yoga improves fitness <comma> lowers blood pressure

# Metrics

Let s be the source string and p your prediction. The quality of the results will be measured according to the following metric:

1.  look for the longest substring w between s and p
2.  compute |w|/max(|s|,|p|)

If the match is exact, the score is 1.

When computing the score, you should NOT consider the start and end tokens.



The longest common substring can be computed with the SequenceMatcher function of difflib, that allows a simple definition of our metric.

In [ ]:
from difflib import SequenceMatcher

def score(s,p):
  match = SequenceMatcher(None, s, p).find_longest_match()
  return (match.size/max(len(p),len(s)))

Let's do an example.

In [ ]:
original = "at first henry wanted to be friends with the king of france"
generated = "henry wanted to be friends with king of france at the first"
print("your score is ",score(original,generated))

your score is  0.5423728813559322


The score must be computed as an average of at least 3K random examples taken form the test set.

# What to deliver

You are supposed to deliver a single notebook, suitably commented.
The notebook should describe a single model, although you may briefly discuss additional attempts you did.

The notebook should contain a full trace of the training.
Weights should be made available on request.

You must also give a clear assesment of the performance of the model, computed with the metric that has been given to you.

# Good work!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from keras.layers import LayerNormalization, MultiHeadAttention, Add, Conv1D, Input, Dense, Embedding
from keras.layers import GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, LayerNormalization, Dropout
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

In [ ]:
### Positional Encoding ###
def positional_encoding(max_sequence_len, model_dim):
    """
    :param max_sequence_len: the maximum length of the sequence
    :param model_dim: the embedding dimension
    """

    positions = np.arange(max_sequence_len)[:, np.newaxis]
    dimensions = np.arange(model_dim)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(model_dim))
    angle_rads = positions * angle_rates

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return pos_encoding

In [ ]:
### Transformer Block ###
def transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate):
    """
    :param latent_dim: the embedding dimension
    :param num_heads: the number of heads in the multiheadattention models
    :param feed_forward_dim: the number of neurons in the feedforward network
    :param dropout_rate: the dropout rate
    """

    input_layer = Input(shape=(None, latent_dim))

    # Multi-head attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    attention_output = attention(input_layer, input_layer, input_layer)
    attention_output = Dropout(dropout_rate)(attention_output)
    attention_output = Add()([input_layer, attention_output])
    attention_output = LayerNormalization()(attention_output)

    # Feedforward
    outputs = Dense(feed_forward_dim, activation='relu')(attention_output)
    outputs = Dense(latent_dim)(outputs)
    outputs = Dropout(dropout_rate)(outputs)
    outputs = Add()([attention_output, outputs])
    outputs = LayerNormalization()(outputs)

    return Model(input_layer, outputs)

In [ ]:
### Transformer Model ###
def transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size):
    """
    :param latent_dim: the embedding dimension
    :param num_heads: the number of heads in the multiheadattention models
    :param feed_forward_dim: the number of neurons in the feedforward network
    :param dropout_rate: the dropout rate
    :param max_sequence_len: the maximum sequence length
    :param num_transformer_blocks: the number of transformer blocks
    :param vocab_size: the size of the vocabulary
    """

    # Encoder
    encoder_inputs = Input(shape=(max_sequence_len,))
    encoder_embedding = Embedding(vocab_size, latent_dim)(encoder_inputs)
    pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    encoder_outputs = encoder_embedding + pos_encoding

    for i in range(num_transformer_blocks):
        encoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(encoder_outputs)

    # Decoder
    decoder_inputs = Input(shape=(max_sequence_len,))
    decoder_embedding = Embedding(vocab_size, latent_dim)(decoder_inputs)
    pos_encoding = positional_encoding(max_sequence_len, latent_dim)
    decoder_outputs = decoder_embedding + pos_encoding

    for i in range(num_transformer_blocks):
        decoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(decoder_outputs)

    # Attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim // num_heads, dropout=dropout_rate)
    context_vector = attention(decoder_outputs, encoder_outputs, encoder_outputs)
    decoder_combined_context = Concatenate()([context_vector, decoder_outputs])
    
    # Re-adapt dimension
    decoder_combined_context = Dense(latent_dim, activation='relu')(decoder_combined_context) 

    # Transformer block
    decoder_outputs = transformer_block(latent_dim, num_heads, feed_forward_dim, dropout_rate)(decoder_combined_context)

    # Output layer
    decoder_outputs = Dense(vocab_size, activation='softmax')(decoder_outputs)
    
    return Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [ ]:
latent_dim = 64
num_heads = 4
feed_forward_dim = 4
dropout_rate = 0.5
max_sequence_len = 28
num_transformer_blocks = 4
vocab_size = len(tokenizer.get_vocabulary())

model = transformer_model(latent_dim, num_heads, feed_forward_dim, dropout_rate, max_sequence_len, num_transformer_blocks, vocab_size)
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy')
model.summary()

Model: "model_49"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_50 (InputLayer)       [(None, 28)]                 0         []                            
                                                                                                  
 input_45 (InputLayer)       [(None, 28)]                 0         []                            
                                                                                                  
 embedding_9 (Embedding)     (None, 28, 128)              1280000   ['input_50[0][0]']            
                                                                                                  
 embedding_8 (Embedding)     (None, 28, 128)              1280000   ['input_45[0][0]']            
                                                                                           

In [ ]:
train_generator = DataGenerator(original_data[:200000], training=True, batch_size=32)
val_generator = DataGenerator(original_data[200000:220000], training=False, batch_size=32)
test_generator = DataGenerator(original_data[220000:], training=False, batch_size=32)

In [71]:
# FIRST TRAINING - 5 EPOCHS ####
checkpoint_path = "/content/drive/MyDrive/Colab/PRJ/transformer8_128_32_8.h5"
checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

history1 = model.fit(train_generator, epochs=5, callbacks=[checkpoint, early_stopping], validation_data=val_generator)
# model.save(checkpoint_path)

Epoch 1/5
6250/6250 [==============================] - ETA: 0s - loss: 0.6213

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


6250/6250 [==============================] - 330s 49ms/step - loss: 0.6213 - val_loss: 16.5937
Epoch 2/5
6250/6250 [==============================] - 298s 48ms/step - loss: 0.0448 - val_loss: 20.9697
Epoch 3/5
6250/6250 [==============================] - 299s 48ms/step - loss: 0.0249 - val_loss: 22.8789
Epoch 4/5
6250/6250 [==============================] - 298s 48ms/step - loss: 0.0176 - val_loss: 24.4913


In [ ]:
### SECOND TRAINING - 5 EPOCHS ###
from tensorflow.keras.models import load_model

history2 = model.fit(train_generator, epochs=5, validation_data=val_generator, callbacks=[checkpoint, early_stopping])
# model.save('/content/drive/MyDrive/Colab/PRJ/transformer8_128_32_8_10epochs.h5')

Epoch 1/5
6250/6250 [==============================] - 286s 46ms/step - loss: 0.0146
Epoch 2/5
6250/6250 [==============================] - 279s 45ms/step - loss: 0.0133
Epoch 3/5
3479/6250 [===============>..............] - ETA: 2:03 - loss: 0.0116

In [72]:
### Show Predicted Sentences ###

x, y_true = test_generator.__getitem__(1)
y_pred = model.predict(x)

# Get tokens with highest probability
y_pred = np.argmax(y_pred, axis=-1)

x = detokenizer(x[0])
y_true = detokenizer(y_true)
y_pred = detokenizer(y_pred)

for i in range(7):
  print("shuffled: ", x[i])
  print("original: ", y_true[i])
  print("predicted: ", y_pred[i])
  print("\n")

1/1 [==============================] - 1s 950ms/step
shuffled:  <start> respiratory <comma> tuberculosis <comma> wildlife <comma> and skin diseases rabies spread can distemper problems <end>
original:  <start> wildlife can spread rabies <comma> skin diseases <comma> tuberculosis <comma> distemper and respiratory problems <end>
predicted:  respiratory <comma> tuberculosis <comma> wildlife <comma> and skin diseases rabies spread can distemper problems <end> <start>


shuffled:  <start> benefits economic social provide people <comma> trees for and numerous environmental <end>
original:  <start> trees provide numerous environmental <comma> social and economic benefits for people <end>
predicted:  benefits economic social provide people <comma> trees for and numerous environmental <end> <start>


shuffled:  <start> red attack stem rot many fungi trunk diseases maple and <end>
original:  <start> many trunk rot fungi and stem diseases attack red maple <end>
predicted:  red attack stem rot man

In [ ]:
import random
def evaluate_baseline(test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        x, y_true = test_generator.__getitem__(i)
        shuffled = x[0].copy()
        random.shuffle(shuffled)

        # loop because each item is a batch
        for true, shuffle in zip(detokenizer(y_true), detokenizer(shuffled)):
            scores.append(score_func(true, shuffle))
    return np.mean(scores), np.std(scores)

baseline_mean, baseline_std = evaluate_baseline(test_generator, score)
print (f'Baseline score: {baseline_mean}')
print (f'Baseline Stdev: {baseline_std}')
print (f'Baseline 3*Stdev: {baseline_std * 3}')

In [ ]:
def evaluate_model(model, test_generator, score_func):
    scores = []

    for i in range(len(test_generator)):
        print("Test :", i)
        x, y_true = test_generator.__getitem__(i)
        y_pred = model.predict(x)
        y_pred = np.argmax(y_pred, axis=-1)

        # loop because each item is a batch
        for true, pred in zip(detokenizer(y_true), detokenizer(y_pred)): 
              scores.append(score_func(true, pred))
    return np.mean(scores), np.std(scores)

average_score, stdev_score = evaluate_model(model, test_generator, score)
print(f'Average score: {average_score}')
print(f'Stdev: {stdev_score}')
print(f'3*Stdev: {stdev_score * 3}')

In [ ]:

# compute the improvement over random guess
improvement = (average_score - baseline_mean) / baseline_mean
print(f'Improvement over random guess: {improvement}')

Random guess: 0.0001
Improvement over random guess: 5238.946940314587
